In [5]:
import pandas as pd
import spacy
import random
from collections import Counter
import matplotlib.pyplot as plt

df = pd.read_csv("gutenberg_novels_dataset.csv")



In [6]:
# Tabla inicial con identificación de los libros
books_info = df[["title", "author", "text_length"]].copy()

# Agregamos el año aproximado de publicación manualmente
publication_years = {
    "Pride and Prejudice": "1813",
    "Frankenstein; Or, The Modern Prometheus": "1818",
    "Dracula": "1897"
}

books_info["publication_year"] = books_info["title"].map(publication_years)

books_info

,title,author,text_length,publication_year
0,Pride and Prejudice,Jane Austen,728392,1813
1,Frankenstein,Mary Shelley,419290,NaN
2,Dracula,Bram Stoker,845805,1897


## 2. Procesamiento con spaCy

Usaremos spaCy porque permite hacer segmentación en oraciones, tokenización, POS tagging y NER con el mismo modelo.

In [8]:
# Cargamos el modelo de spaCy en inglés

nlp = spacy.load("en_core_web_sm")

# Aumentamos el límite máximo de caracteres para poder procesar textos largos
nlp.max_length = 2_000_000

In [9]:
# Diccionario donde guardaremos la información procesada de cada libro
books = {}

# Procesamos cada novela por separado
for _, row in df.iterrows():
    title = row["title"]
    author = row["author"]
    text = row["text"]
    
    # Procesamos el texto completo con spaCy
    doc = nlp(text)
    
    # Segmentación en oraciones
    sentences = list(doc.sents)
    
    # Tokenización: quitamos espacios vacíos
    tokens = [token for token in doc if not token.is_space]
    
    # Seleccionamos una muestra aleatoria de 100 oraciones
    sample_sentences = random.sample(sentences, 100)
    
    # Extraemos los tokens de la muestra
    sample_tokens = [
        token
        for sent in sample_sentences
        for token in sent
        if not token.is_space
    ]
    
    # Guardamos todo en el diccionario
    books[title] = {
        "author": author,
        "doc": doc,
        "sentences": sentences,
        "tokens": tokens,
        "sample_sentences": sample_sentences,
        "sample_tokens": sample_tokens
    }

In [ ]:
# Creamos la tabla comparativa 
summary_rows = []

for title, data in books.items():
    summary_rows.append({
        "Libro": title,
        "Autor": data["author"],
        "Oraciones totales": len(data["sentences"]),
        "Tokens totales": len(data["tokens"]),
        "Oraciones en muestra": len(data["sample_sentences"]),
        "Tokens en muestra": len(data["sample_tokens"])
    })

summary_df = pd.DataFrame(summary_rows)

summary_df

,Libro,Autor,Oraciones totales,Tokens totales,Oraciones en muestra,Tokens en muestra
0,Pride and Prejudice,Jane Austen,5754,152569,100,2516
1,Frankenstein,Mary Shelley,3250,85907,100,2491
2,Dracula,Bram Stoker,8444,191556,100,2266
